In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')

The API key found and looks good so far!


# CodeEvolve

Chapter 9 ("Learning and Adaptation", pp. 135-146) closes with AlphaEvolve and OpenEvolve: agents that *learn* without touching the model's weights. A population of programs is scored by an automated evaluator, an LLM proposes edits to the best ones, and the population drifts toward better scores. The book's listing is OpenEvolve's `evolve.run(iterations=1000)`. This notebook runs the same pattern with **CodeEvolve**, an open-source framework whose authors published everything needed to reproduce their results:

> Assumpcao, Ferreira, Campos, Murai (2025). *CodeEvolve: an open-source evolutionary framework for algorithmic discovery and optimization.* arXiv:2510.14150. Code: https://github.com/inter-co/science-codeevolve

## The pattern in one minute

Chapters 1-8 improved an answer within one conversation. Here the thing that improves is a **population of programs**, and the model is used as a mutation operator: it reads a parent program and its score and proposes a targeted edit. An evaluator runs the child in a sandbox and scores it; selection keeps the fittest. Nothing inside the LLM changes. The adaptation lives in three growing populations:

| Population | What it holds | Who writes it |
|---|---|---|
| solutions | programs with their metrics and parent pointers | the evaluator |
| prompts | system prompts, scored by the best child they produced | an auxiliary "meta-prompting" LLM |
| MAP-Elites archive | the best program per region of a feature space (here: fitness x runtime) | the evaluator |

## The five ideas in the paper

1. **Islands.** $N$ populations evolve in parallel processes and swap their best programs every few epochs along a ring. Diversity comes from isolation; good ideas spread by migration.
2. **Two operators, one coin flip per epoch.** With probability $1 - p_{explr}$, *depth exploitation*: pick a parent by rank-based roulette, show the model the parent, its ancestors as chat history, and $\iota$ "inspiration" programs, and ask for an incremental edit. Otherwise *meta-prompting exploration*: pick a random parent and a random prompt, let an auxiliary LLM rewrite the prompt, and generate from the rewritten prompt with no ancestors. A plateau scheduler raises $p_{explr}$ when the best score stalls.
3. **Inspiration-based crossover.** Classic crossover splices two programs; with diff edits that is unsafe. Instead the model sees several strong programs next to the parent and transfers their ideas as edits, which makes each step effectively multi-parent.
4. **SEARCH/REPLACE edits.** The model never rewrites the file. It returns blocks of `<<<<<<< SEARCH` / `=======` / `>>>>>>> REPLACE`, applied only inside `# EVOLVE-BLOCK-START` / `END` markers. A block that does not match the parent exactly is rejected and the epoch produces nothing. The rejection rate is a property of the model, and part of what this notebook measures.
5. **Prompt fitness.** A prompt's score is the *maximum* fitness among the programs it generated, so prompts evolve with the same machinery as programs.

Headline results: on the AlphaEvolve benchmark the framework matches or beats AlphaEvolve's reported numbers on 5 of 9 problems, and the open-weight Qwen3-Coder-30B does it at about one tenth of the cost of a Gemini ensemble (6 USD against 35 USD on circle packing).

## The task used here: a heuristic for the Travelling Salesman Problem

The paper's second benchmark suite (EoH, Section 5.3) is *automatic heuristic design*: only one function inside a fixed solver is evolved. The solver is Guided Local Search for the TSP. After each local optimum it calls

```python
update_edge_distance(edge_distance, local_opt_tour, edge_n_used) -> updated_edge_distance
```

to make some edges more expensive so the next local search is pushed elsewhere. Only that function is evolved. Fitness is the negative mean gap in percent to the optimal tour over 3 random 100-city instances (10 s each), so 0 is perfect. The paper's result on the TSP100 test set: EoH 0.025%, CodeEvolve 0.040% best and 0.101% mean, with 10 islands for 152 epochs. Here: 2 islands for 10 epochs, about 25 model calls, a few minutes.

Why this task and not circle packing: the evaluator runs in seconds, the starting heuristic is one readable function, and TSP is familiar.

## How this notebook drives CodeEvolve

CodeEvolve is a dependency of this repository (it is why the project needs Python 3.13), and the TSP-GLS problem, the run configuration and the held-out test data are copied into `chapter-9-codeevolve/` next to this notebook (`NOTICE.md` there lists the sources). Nothing outside this repository is needed.

The small steps, applying a diff and building a prompt, run in this kernel. The evolutionary run itself does not: CodeEvolve starts one process per island with the `fork` start method, which is unsafe inside a Jupyter kernel, so the run is launched as a subprocess with the same Python, exactly as from a terminal, and the notebook reads the files it leaves behind. The only extra prerequisite is the model for the Ollama cell: `ollama pull qwen3-coder:30b` (19 GB).

In [3]:
# Shared, model-free cell: the paths and helpers every later cell uses. Nothing here calls a model,
# so the Ollama cell runs without the hosted one.
import difflib
import json
import pickle
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from uuid import uuid4

import codeevolve
import numba
import pandas as pd
import yaml
from codeevolve.database import Program, ProgramDatabase
from codeevolve.prompt.sampler import PromptSampler, format_prog_msg
from codeevolve.prompt.template import format_eval_budget
from codeevolve.utils.parsing import apply_diff

HERE = Path("chapter-9-codeevolve").resolve()        # copied from CodeEvolve, see NOTICE.md there
PROBLEM = HERE / "tsp_gls/input"                     # initial program, evaluate.py and the GLS solver
TEST_SCRIPT = HERE / "tsp_gls/test_evaluate.py"
TEST_DATA = HERE / "tsp_testing"
LAUNCHER = HERE / "run_codeevolve.py"                # makes the CLI work on macOS; a no-op on Linux
CFG_TEMPLATE = HERE / "config_tsp_gls.yaml"          # the paper's TSP config, budget reduced
OUT = Path("generated/codeevolve"); OUT.mkdir(parents=True, exist_ok=True)


def evaluate(program_path):
    '''Score one heuristic file with the problem's evaluator; returns the metrics dict.'''
    results = OUT / (Path(program_path).stem + ".json")
    subprocess.run([sys.executable, str(PROBLEM / "evaluate.py"), str(Path(program_path).resolve()), str(results.resolve())],
                   cwd=PROBLEM, check=True, capture_output=True)
    return json.load(open(results))


def run_codeevolve(cfg_path, out_dir, api_base, api_key):
    '''Run one CodeEvolve experiment to completion (blocking) and return the output directory.'''
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    cmd = [sys.executable, str(LAUNCHER), f"--inpt_dir={PROBLEM}/", f"--cfg_path={Path(cfg_path).resolve()}",
           f"--out_dir={out_dir.resolve()}/", "--load_ckpt=0"]
    t0 = time.time()
    res = subprocess.run(cmd, cwd=HERE, input="y\n", capture_output=True, text=True,
                         env={**os.environ, "API_BASE": api_base, "API_KEY": api_key})
    (out_dir / "run.log").write_text(res.stdout + res.stderr)
    print(f"finished in {(time.time() - t0) / 60:.1f} min, exit code {res.returncode}")
    print(res.stdout[res.stdout.rfind("GLOBAL BEST SOLUTION"):][:600])
    return out_dir


def epoch_table(out_dir):
    '''Per-epoch child fitness and running best for every island, parsed from the island logs.'''
    rows = {}
    for log in sorted(Path(out_dir).glob("island_*/logs/run_*.log")):
        isl = log.parts[-3]
        best = None
        for epoch, block in re.findall(r"EPOCH (\d+) =+(.*?)(?=EPOCH \d+ =|ALGORITHM FINISHED)", log.read_text(), re.S):
            m = re.search(r"Child solution -> Program\(id=[^,]+,fitness=(-?[\d.]+)", block)
            child = float(m.group(1)) if m else None
            if child is not None and child > -1e6:
                best = child if best is None else max(best, child)
            rows.setdefault(int(epoch), {})[f"{isl} child"] = child
            rows[int(epoch)][f"{isl} best"] = best
    return pd.DataFrame.from_dict(rows, orient="index").sort_index().rename_axis("epoch")


def best_heuristic(out_dir):
    '''The best program across islands: (island dir, code, metrics from run_metadata.json).'''
    meta = json.load(open(Path(out_dir) / "run_metadata.json"))
    last = meta[str(max(int(k) for k in meta))]["best_sol"]
    isl_dir = Path(out_dir) / f"island_{last['island_found']}"
    return isl_dir, (isl_dir / "best_sol.py").read_text(), last


def held_out_test(program_path, max_instances=100):
    '''Gap (%) on EoH's held-out TSP20 set and on 61 held-out TSP100 instances.'''
    res = subprocess.run([sys.executable, "-I", str(TEST_SCRIPT), str(Path(program_path).resolve()),
                          "--test-data-dir", str(TEST_DATA), "--max-instances", str(max_instances)],
                         cwd=TEST_SCRIPT.parent, capture_output=True, text=True)
    gaps = [l for l in res.stdout.splitlines() if l.startswith("| **CodeEvolve**")]
    return gaps[0] if gaps else res.stderr[-400:]


print("codeevolve", codeevolve.__version__, "| numba", numba.__version__, "| python", sys.version.split()[0])

codeevolve 0.3.1 | numba 0.68.0 | python 3.13.12


## The problem: one function inside a fixed solver

A CodeEvolve *problem* is a directory with an initial program containing an evolve block, and an `evaluate.py` that the framework calls as `python evaluate.py <candidate.py> <results.json>`. The framework knows nothing about tours; it only reads the JSON of metrics and takes `fitness`. The initial heuristic is the textbook linear penalty.

In [4]:
print((PROBLEM / "src/init_program.py").read_text())
init_metrics = evaluate(PROBLEM / "src/init_program.py")
print(init_metrics)

# EVOLVE-BLOCK-START
import numpy as np


def update_edge_distance(edge_distance, local_opt_tour, edge_n_used):
    """Update edge distances to escape local optima in Guided Local Search for TSP.

    Args:
        edge_distance: np.ndarray of shape (n, n) - original edge distance matrix.
        local_opt_tour: np.ndarray of node IDs in the current local optimal tour.
        edge_n_used: np.ndarray of shape (n, n) - count of how many times each
            edge has been used during perturbation (penalty matrix).

    Returns:
        updated_edge_distance: np.ndarray of shape (n, n) - modified distance matrix
            that guides the local search away from the current local optimum.
    """
    updated_edge_distance = edge_distance.copy()
    n = len(local_opt_tour)
    for i in range(n - 1):
        a = local_opt_tour[i]
        b = local_opt_tour[i + 1]
        updated_edge_distance[a, b] *= (1 + edge_n_used[a, b])
        updated_edge_distance[b, a] = updated_edge_distance[a, b

{'fitness': -4.582259289995391, 'avg_gap_pct': 4.582259289995391, 'gaps': [6.981603922675039, 3.957301921524947, 2.8078720257861844], 'eval_time': 1.093153953552246}


## The edit format the model must produce

Every generation is a SEARCH/REPLACE diff applied by `codeevolve.utils.parsing.apply_diff`. Before the model does it, do it by hand, because it shows what the solver actually rewards. Reading `gls/gls_evol.py`: the solver subtracts the original matrix from the heuristic's output, penalises the **five edges with the largest increase**, and runs local search around them. Only the ranking matters, and `edge_n_used` is all zeros on the first call, so the initial rule changes nothing at first and the solver starts by penalising arbitrary edges. That is why its gap is a poor 4.6%.

One line that injects a real signal, the utility idea of the original GLS paper: add the normalised edge length to the count, so long tour edges are penalised first.

In [5]:
parent = (PROBLEM / "src/init_program.py").read_text()
diff = (
    "<<<<<<< SEARCH\n"
    "    updated_edge_distance = edge_distance.copy()\n"
    "=======\n"
    "    updated_edge_distance = edge_distance.copy()\n"
    "    edge_n_used = edge_n_used + edge_distance / edge_distance.mean()\n"
    + ">" * 7 + " REPLACE\n"          # assembled this way because IPython strips a leading '>>> ' from cell lines
)
(OUT / "diff.txt").write_text(diff)
child = apply_diff(parent, diff)
(OUT / "length_aware.py").write_text(child)
print(diff)
print("initial rule     :", init_metrics["fitness"])
print("length-aware rule:", evaluate(OUT / "length_aware.py")["fitness"])

<<<<<<< SEARCH
    updated_edge_distance = edge_distance.copy()
    updated_edge_distance = edge_distance.copy()
    edge_n_used = edge_n_used + edge_distance / edge_distance.mean()
>>>>>>> REPLACE

initial rule     : -4.582259289995391


length-aware rule: -0.4882146797179243


Keep that number in mind. One line of domain knowledge took the gap from 4.6% to about 0.5%. The question the run below answers is whether the model finds something as good, and how many generations it needs.

## What the model sees

`PromptSampler.build()` assembles the chat. The system message is the evolvable `SYS_MSG` from the config (the paper's own TSP prompt, with its design tips) plus the evaluation budget and the task rules. The user turn carries the parent program with its metrics. In exploitation mode the ancestors are replayed as earlier turns and inspiration programs are prepended.

In [6]:
cfg = yaml.safe_load(open(CFG_TEMPLATE))
sampler = PromptSampler(aux_lm_cfg={"model_name": "MOCK"}, api_key="x", api_base="x")
db = ProgramDatabase(id=0, seed=0, max_alive=None, elite_map_type=None, features=None)
prompt = Program(id=str(uuid4()), code=cfg["SYS_MSG"], language="text")
sol = Program(id=str(uuid4()), code=parent, language="python", returncode=0, eval_metrics=init_metrics)
sol.prog_msg = format_prog_msg(sol)
db.add(sol)
budget = format_eval_budget(cfg["BUDGET_CONFIG"]["eval_timeout"], cfg["BUDGET_CONFIG"]["max_mem_bytes"])
for m in sampler.build(prompt=prompt, prog=sol, db=db, max_chat_depth=0, exploitation=False, eval_budget=budget):
    print("===", m["role"].upper(), f"({len(m['content'])} chars) ===")
    print(m["content"][:2200] + ("\n[...]" if len(m["content"]) > 2200 else ""), "\n")

=== SYSTEM (7165 chars) ===
# PROMPT-BLOCK-START
You are an expert in combinatorial optimization, metaheuristics, and the Travelling Salesman Problem (TSP).
Your mission is to design a novel and high-performing edge distance update strategy for
Guided Local Search (GLS) applied to TSP.

PROBLEM CONTEXT:
- **Task**: Design a function that updates the edge distance matrix to guide the local
  search away from the current local optimum. The GLS framework iteratively applies 2-opt
  and relocate local search operators, using your modified distance matrix to escape local
  optima and explore new regions of the solution space.
- **Function signature**: `update_edge_distance(edge_distance, local_opt_tour, edge_n_used) -> updated_edge_distance`
- **Inputs**:
  * `edge_distance` (np.ndarray, shape (n, n)): The original symmetric distance matrix between cities.
  * `local_opt_tour` (np.ndarray): Array of node IDs representing the current locally optimal tour.
  * `edge_n_used` (np.ndarray, shape

## Configure a run

One YAML file holds the problem, the budget, the evolutionary parameters and the models. The template is the paper's TSP configuration with the budget reduced; the cell shrinks it further for a live run and leaves the model name to be filled in by each run cell.

| Config key | Paper concept |
|---|---|
| `num_islands`, `migration.topology/interval/rate` | islands, elitist migration along a ring |
| `exploration_rate`, `exploration_scheduler` | the exploit/explore coin, the plateau scheduler |
| `max_chat_depth` ($k$), `num_inspirations` ($\iota$) | depth exploitation, inspiration-based crossover |
| `meta_prompting`, `SAMPLER_AUX_LM` | meta-prompting exploration |
| `MAP_ELITES` (CVT over fitness and eval_time) | quality-diversity archive |
| `BUDGET_CONFIG` | sandbox time and memory limits, also injected into the prompt |
| `init_pop` | how many children of the initial program are generated before selection starts |

In [7]:
cfg["EVOLVE_CONFIG"].update(num_islands=2, num_epochs=10, init_pop=3, ckpt=5)
cfg["EVOLVE_CONFIG"]["migration"]["interval"] = 5
print(cfg["EVOLVE_CONFIG"])


def write_config(model_name, path):
    c = json.loads(json.dumps(cfg))
    for key in ("EXPLORATION_ENSEMBLE", "EXPLOITATION_ENSEMBLE"):
        c[key][0]["model_name"] = model_name
    c["SAMPLER_AUX_LM"]["model_name"] = model_name
    yaml.safe_dump(c, open(path, "w"), sort_keys=False)
    return path

{'fitness_key': 'fitness', 'num_epochs': 10, 'ckpt': 5, 'max_size': None, 'init_pop': 3, 'exploration_rate': 0.2, 'selection': {'policy': 'roulette', 'kwargs': {'roulette_by_rank': True}}, 'early_stopping_rounds': None, 'num_islands': 2, 'migration': {'topology': 'ring', 'interval': 5, 'rate': 0.1}, 'meta_prompting': True, 'use_map_elites': True, 'num_inspirations': 2, 'max_chat_depth': 3, 'exploration_scheduler': {'type': 'PlateauScheduler', 'kwargs': {'min_value': 0.2, 'max_value': 0.5, 'plateau_threshold': 5, 'increase_factor': 1.05, 'decrease_factor': 0.95}}}


## Run CodeEvolve on OpenAI

CodeEvolve speaks to any OpenAI-compatible endpoint through `API_BASE` and `API_KEY`. `gpt-4.1-mini` follows the edit format reliably (3 of 3 probes) at about 5 seconds a call; `gpt-4o-mini` returned a reply without any diff block in 2 of 4 probes, which the framework counts as a wasted epoch. Twenty epochs over two islands cost a few cents. Expect 3 to 5 minutes: the model calls are fast, the GLS evaluations are 2 to 30 seconds each.

In [8]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set; skip to the Ollama cell")

HOSTED_MODEL = "gpt-4.1-mini"
hosted_out = run_codeevolve(write_config(HOSTED_MODEL, OUT / "config_openai.yaml"), OUT / "run_openai",
                            api_base="https://api.openai.com/v1", api_key=os.environ["OPENAI_API_KEY"])
display(epoch_table(hosted_out))

finished in 4.7 min, exit code 0
GLOBAL BEST SOLUTION = GlobalBestProg(
  fitness=-0.02457024,
  iteration_found=10,
  island_found=1,
  depth=4,
  eval_metrics={
    fitness: -0.024570244043324124,
    avg_gap_pct: 0.024570244043324124,
    gaps: [-2.220446049250313e-14, 0.07371073213002788, -3.3306690738754696e-14],
    eval_time: 30.0208797454834,
  }
)
> GLOBAL EARLY STOPPING COUNTER = 0
============================================= FINISHED =============================================



,island_0 child,island_0 best,island_1 child,island_1 best
epoch,,,,
1,-4.582259,-4.582259,-2.621761,-2.621761
2,-2.916762,-2.916762,-2.375489,-2.375489
3,NaN,-2.916762,-1.231237,-1.231237
4,NaN,-2.916762,-1.951531,-1.231237
5,-2.916762,-2.916762,-0.961401,-0.961401
6,-4.582259,-2.916762,-4.531381,-0.961401
7,-4.582259,-2.916762,-0.720427,-0.720427
8,-4.582259,-2.916762,-0.689541,-0.689541
9,NaN,-2.916762,-0.068499,-0.068499


Read the table top to bottom: the first `init_pop` epochs are children of the initial program, selection starts after that, and a `None` child is an epoch whose edit was rejected or whose program crashed. The run's best program and the prompt that produced it are plain files in the island directory.

In [9]:
isl_dir, code_text, meta = best_heuristic(hosted_out)
print(f"best heuristic from {isl_dir.name}: fitness {meta['fitness']:.4f} (mean gap {-meta['fitness']:.3f}%), "
      f"found at epoch {meta['iteration_found']}, lineage depth {meta['depth']}")
print(code_text)
prompt_diff = "".join(difflib.unified_diff(cfg["SYS_MSG"].splitlines(True), (isl_dir / "best_prompt.txt").read_text().splitlines(True),
                                           "initial SYS_MSG", "best evolved prompt", n=0))
print(prompt_diff or "the island's best prompt is still the initial one")

best heuristic from island_1: fitness -0.0246 (mean gap 0.025%), found at epoch 10, lineage depth 4
# EVOLVE-BLOCK-START
import numpy as np

def update_edge_distance(edge_distance, local_opt_tour, edge_n_used):
    """Update edge distances to escape local optima in Guided Local Search for TSP.

    Improved approach combining adaptive nonlinear penalties on tour edges weighted by normalized edge length,
    dynamic penalty caps, softened diffusion of penalties to neighboring edges with deterministic jitter,
    and a mild capped logarithmic penalty on non-tour edges to encourage exploration while controlling penalty growth.
    Includes a small deterministic spectral component to enhance structural penalty guidance.
    """
    n = len(local_opt_tour)
    updated_edge_distance = edge_distance.copy()

    # Get tour edges indices (from, to)
    idx_from = local_opt_tour
    idx_to = np.roll(local_opt_tour, -1)

    # Penalties and distances on tour edges
    usage = edge_n_used[idx_from

## Did it generalise?

Evolution scored each heuristic on only 3 instances. `test_evaluate.py` (in `chapter-9-codeevolve/tsp_gls/`) runs a heuristic on held-out sets: EoH's published TSP20 file (1000 instances) and 61 TSP100 instances the paper's authors shipped for training but never used during evolution. Columns are TSP20, TSP50 (no test file, so `-`), TSP100.

In [10]:
# build the TSP100 held-out set once: training instances 3..63 of the problem's own TSPAEL64.pkl (not a download), in EoH's format
if not (TEST_DATA / "TSP100.pkl").exists():
    train = pickle.load(open(PROBLEM / "TrainingData/TSPAEL64.pkl", "rb"))
    pickle.dump({k: v[3:] for k, v in train.items()}, open(TEST_DATA / "TSP100.pkl", "wb"))
print("initial rule      ", held_out_test(PROBLEM / "src/init_program.py"))
print("length-aware rule ", held_out_test(OUT / "length_aware.py"))
print(f"{HOSTED_MODEL:18s}", held_out_test(isl_dir / "best_sol.py"))

initial rule       | **CodeEvolve** | **0.971** | - | **3.440** |


length-aware rule  | **CodeEvolve** | **0.053** | - | **1.074** |


gpt-4.1-mini       | **CodeEvolve** | **-0.000** | - | **0.217** |


## CodeEvolve + Ollama

Same run on a local model served by Ollama. The paper's open-weight backbone, Qwen3-Coder-30B, is a mixture-of-experts model with 3B active parameters, so it runs at 50-70 tokens per second on Apple Silicon or a 24 GB GPU. Smaller chat models fail the format: `qwen2.5:7b` returns whole files in code fences and `qwen2.5:32b` writes `<<<<< SEARCH` with five angle brackets, so neither produces a single evaluated child. Prerequisite: `ollama pull qwen3-coder:30b`. About 11 minutes for the same 2 x 10 budget.

In [11]:
OLLAMA_MODEL = "qwen3-coder:30b"
ollama_out = run_codeevolve(write_config(OLLAMA_MODEL, OUT / "config_ollama.yaml"), OUT / "run_ollama",
                            api_base="http://localhost:11434/v1", api_key="ollama")
display(epoch_table(ollama_out))
isl_dir_o, code_text_o, meta_o = best_heuristic(ollama_out)
print(f"best heuristic from {isl_dir_o.name}: fitness {meta_o['fitness']:.4f}, found at epoch {meta_o['iteration_found']}, depth {meta_o['depth']}")
print(code_text_o)
print(f"{OLLAMA_MODEL:18s}", held_out_test(isl_dir_o / "best_sol.py"))

finished in 11.0 min, exit code 0
GLOBAL BEST SOLUTION = GlobalBestProg(
  fitness=-1.83567294,
  iteration_found=8,
  island_found=0,
  depth=2,
  eval_metrics={
    fitness: -1.8356729431028802,
    avg_gap_pct: 1.8356729431028802,
    gaps: [2.369253749860478, 1.7620573972310982, 1.3757076822170644],
    eval_time: 1.8797268867492676,
  }
)
> GLOBAL EARLY STOPPING COUNTER = 2
============================================= FINISHED =============================================



,island_0 child,island_0 best,island_1 child,island_1 best
epoch,,,,
1,-2.837128,-2.837128,-4.582259,-4.582259
2,NaN,-2.837128,-4.582259,-4.582259
3,-4.582259,-2.837128,-2.137524,-2.137524
4,-4.582259,-2.837128,-4.582259,-2.137524
5,-4.582259,-2.837128,-4.582259,-2.137524
6,-4.582259,-2.837128,-4.582259,-2.137524
7,-4.582259,-2.837128,-4.582259,-2.137524
8,-1.835673,-1.835673,-4.582259,-2.137524
9,-4.582259,-1.835673,-4.582259,-2.137524


best heuristic from island_0: fitness -1.8357, found at epoch 8, depth 2
# EVOLVE-BLOCK-START
import numpy as np


def update_edge_distance(edge_distance, local_opt_tour, edge_n_used):
    """Update edge distances to escape local optima in Guided Local Search for TSP.

    This implementation uses a refined Voudouris & Tsang penalty strategy with:
    1. Simple but effective penalty scaling based on usage counts
    2. Tour-specific edge penalties that are more consistent
    3. Balanced logarithmic scaling to avoid over-penalization
    4. Proper handling of the tour structure for memory effects

    Args:
        edge_distance: np.ndarray of shape (n, n) - original edge distance matrix.
        local_opt_tour: np.ndarray of node IDs in the current local optimal tour.
        edge_n_used: np.ndarray of shape (n, n) - count of how many times each
            edge has been used during perturbation (penalty matrix).

    Returns:
        updated_edge_distance: np.ndarray of shape (n, n) 

qwen3-coder:30b    | **CodeEvolve** | **0.042** | - | **1.936** |


## What the paper found at scale, and what to take away

| Heuristic | TSP100 gap (%) | Budget |
|---|---|---|
| initial linear penalty | 3.44 | none |
| one hand-written line (above) | 1.07 | none |
| CodeEvolve, this notebook | 0.22 (gpt-4.1-mini), 1.94 (qwen3-coder:30b) | 2 islands x 10 epochs, 5 and 11 min |
| CodeEvolve, an earlier run of this notebook | 1.33 (gpt-4.1-mini), 1.52 (qwen3-coder:30b) | the same |
| CodeEvolve, course assignment run | 0.51 | 2 x 20, Qwen3-Coder-30B, 26 min on a laptop |
| CodeEvolve, paper (Table 4) | 0.040 best, 0.101 mean | 10 x 152, Qwen3-Coder-30B |
| EoH, reported | 0.025 | GPT-3.5-turbo |

Read the first four rows together. At 10 epochs the outcome is mostly luck: in the committed run the hosted model beat the hand-written line on held-out data, and its winner weights penalties by normalised edge length, the same idea as that line, while the local model did not; in an earlier run neither did. The paper's 1,500 generations are what make the search reliable rather than lucky. The search scales with budget; human insight does not need to.

Three things to draw out in class:

1. **Where the intelligence lives.** The model proposes; the evaluator decides. A weak evaluator (3 instances) is the agent's whole view of the world, and the held-out table is the check on it. This is chapter 11's monitor again, with a program as the judge instead of an LLM.
2. **The search is only as good as the edit channel.** A model that cannot produce an exact SEARCH block wastes the epoch, and the framework cannot tell a bad idea from a bad diff. Compare the rejection counts in the two runs.
3. **Learning without training.** Nothing was fine-tuned. The "memory" of this agent is the population, the prompt pool and the archive on disk; restart the run with `--load_ckpt=-1` and it continues from there. Ask the class which of chapter 9's four kinds of learning this is.

The full course assignment goes further: the mock run, checkpoint analysis, lineage plots, the circle-packing reproduction and the ablation exercises.